# microgpt ラボ — 200行で GPT を育てる / The 200-line GPT Lab

ChatGPT の「超ミニ版」を、たった **200行の Python** で動かします。ライブラリなし。かくしごとなし。全部このノートの中で見えます。

**今日やること / Today**
1. ポケモンの名前 1000 体で AI を訓練する
2. 温度(temperature)のつまみで遊ぶ
3. 民謡・チェス・俳句 — データを変えて「別の専門家」を育てる
4. 自分のデータで「自分GPT」を作る

**準備 / Setup**: ランタイムは **CPU のままでOK**(GPU は使いません)。上のメニューから「ランタイム → すべてのセルを実行」はせず、**1つずつ順番に**実行してください。

> やさしい日本語メモ: 「訓練(くんれん)」= AI にデータをたくさん見せて、まねできるように育てること。

## Step 1 — microgpt.py をこの Colab に置く / Get microgpt.py

下のセルを実行すると、`microgpt.py` がこの Colab に保存されます。これが **全部** です。この 200 行の外に魔法はありません。

(作者: Andrej Karpathy。コースサイトの Day 2 ページにコードの読み方ガイドがあります)

In [ ]:
%%writefile microgpt.py
"""
The most atomic way to train and run inference for a GPT in pure, dependency-free Python.
This file is the complete algorithm.
Everything else is just efficiency.

@karpathy
"""

import os       # os.path.exists
import math     # math.log, math.exp
import random   # random.seed, random.choices, random.gauss, random.shuffle
random.seed(42) # Let there be order among chaos

# Let there be a Dataset `docs`: list[str] of documents (e.g. a list of Pokemon names)
if not os.path.exists('input.txt'):
    # 1) Prefer the dataset bundled next to this script — no network needed.
    here = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
    bundled = os.path.join(here, '..', 'data', 'pokemon.txt')
    if os.path.exists(bundled):
        import shutil
        shutil.copyfile(bundled, 'input.txt')
    else:
        # 2) Otherwise download it. Retry without TLS verification if this Python
        #    has no CA certificates installed (common with Homebrew Python on Mac).
        import ssl, urllib.request
        names_url = 'https://raw.githubusercontent.com/itoksk/summer-ai-materials/main/materials/data/pokemon.txt'
        data, last_error = None, None
        for ctx in (ssl.create_default_context(), ssl._create_unverified_context()):
            try:
                with urllib.request.urlopen(names_url, context=ctx) as r:
                    data = r.read()
                break
            except Exception as err:
                last_error = err
        if data is None:
            raise SystemExit(
                "Could not load the dataset. Put a UTF-8 text file (one item per line) "
                f"named 'input.txt' next to this script and re-run.\nLast error: {last_error}"
            )
        with open('input.txt', 'wb') as f:
            f.write(data)
# Always read as UTF-8 so Japanese names work on every OS (Windows defaults to cp932).
docs = [line.strip() for line in open('input.txt', encoding='utf-8') if line.strip()]
random.shuffle(docs)
print(f"num docs: {len(docs)}")

# Let there be a Tokenizer to translate strings to sequences of integers ("tokens") and back
uchars = sorted(set(''.join(docs))) # unique characters in the dataset become token ids 0..n-1
BOS = len(uchars) # token id for a special Beginning of Sequence (BOS) token
vocab_size = len(uchars) + 1 # total number of unique tokens, +1 is for BOS
print(f"vocab size: {vocab_size}")

# Let there be Autograd to recursively apply the chain rule through a computation graph
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads') # Python optimization for memory usage

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # scalar value of this node calculated during forward pass
        self.grad = 0                   # derivative of the loss w.r.t. this node, calculated in backward pass
        self._children = children       # children of this node in the computation graph
        self._local_grads = local_grads # local derivative of this node w.r.t. its children

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

# Initialize the parameters, to store the knowledge of the model
n_layer = 1     # depth of the transformer neural network (number of layers)
n_embd = 16     # width of the network (embedding dimension)
block_size = 16 # maximum context length of the attention window (note: the longest name is 15 characters)
n_head = 4      # number of attention heads
head_dim = n_embd // n_head # derived dimension of each head
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
params = [p for mat in state_dict.values() for row in mat for p in row] # flatten params into a single list[Value]
print(f"num params: {len(params)}")

# Define the model architecture: a function mapping tokens and parameters to logits over what comes next
# Follow GPT-2, blessed among the GPTs, with minor differences: layernorm -> rmsnorm, no biases, GeLU -> ReLU
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id] # token embedding
    pos_emb = state_dict['wpe'][pos_id] # position embedding
    x = [t + p for t, p in zip(tok_emb, pos_emb)] # joint token and position embedding
    x = rmsnorm(x) # note: not redundant due to backward pass via the residual connection

    for li in range(n_layer):
        # 1) Multi-head Attention block
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]
        # 2) MLP block
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

# Let there be Adam, the blessed optimizer and its buffers
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params) # first moment buffer
v = [0.0] * len(params) # second moment buffer

# Repeat in sequence
num_steps = 1000 # number of training steps
for step in range(num_steps):

    # Take single document, tokenize it, surround it with BOS special token on both sides
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # Forward the token sequence through the model, building up the computation graph all the way to the loss
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses) # final average loss over the document sequence. May yours be low.

    # Backward the loss, calculating the gradients with respect to all model parameters
    loss.backward()

    # Adam optimizer update: update the model parameters based on the corresponding gradients
    lr_t = learning_rate * (1 - step / num_steps) # linear learning rate decay
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end='\r')

# Inference: may the model babble back to us
temperature = 0.5 # in (0, 1], control the "creativity" of generated text, low to high
print("\n--- inference (new, hallucinated names) ---")
for sample_idx in range(20):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")

## Step 2 — ポケモンの名前 1000 体で訓練する / Train on Pokémon names

学習データ `input.txt` は「1行に1つの名前」が並んだだけのファイルです。AI がやるのは **「次の1文字を当てる」** ゲームだけ。それを何百回もくり返すと、名前の「らしさ」を覚えます。

In [ ]:
# ポケモンの名前データをダウンロードして input.txt にする
import urllib.request
url = "https://raw.githubusercontent.com/itoksk/summer-ai-materials/main/materials/data/pokemon.txt"
urllib.request.urlretrieve(url, "input.txt")
lines = open("input.txt").read().splitlines()
print(f"{len(lines)} 行 / lines")
print("\n".join(lines[:10]))

実行中の見方 / How to read the output:
- `loss` = まちがい度。**下がっていけば学んでいる**証拠
- 終わると「図鑑にいない新ポケモン」の名前が 20 個出ます

(数分かかります。loss が動くのをながめるのも勉強のうち)

In [ ]:
!python3 microgpt.py

### 観察タイム / Observe

- どの名前が「ありそう」? 声に出して読んでみよう
- 本物の図鑑にある名前がそのまま出た? それは「創作」ではなく「丸暗記」かも
- となりの人と「ベスト新ポケモン」を1体ずつ選ぼう

## Step 3 — 実験室: つまみをいじる / The experiment lab

ここからは `microgpt_lab.py` を使います。**アルゴリズムは Step 1 と同じ**で、実験用のつまみ(データ・ステップ数・温度など)を外から変えられるようにしただけのものです。

In [ ]:
%%writefile microgpt_lab.py
#!/usr/bin/env python3
"""microgpt_lab.py — microgpt.py(@karpathy)を実験用にパラメータ化したもの。

アルゴリズムは microgpt.py と同一(純Python・依存ライブラリなしの文字レベルGPT)。
違いは「実験のしやすさ」だけ:

  - 学習データ・ステップ数・モデルサイズ・文脈長をコマンドラインで変えられる
  - 1回の訓練で複数の温度(temperature)のサンプルをまとめて出力できる
  - 進捗(loss と経過時間)を定期的に表示する
  - --chat で、訓練後に「双子」と対話できる(文の出だしを与えると続きを書く)

使い方の例:
  python3 microgpt_lab.py --input input.txt
  python3 microgpt_lab.py --input input_haiku.txt --steps 500 --block-size 32
  python3 microgpt_lab.py --input input.txt --temperatures 0.1 0.5 1.0 2.0
  python3 microgpt_lab.py --input input_me.txt --steps 600 --block-size 40 --chat

  # 1回だけ学習して保存 → 次からは読み込んで(学習せずに)チャットだけ
  python3 microgpt_lab.py --input input_me.txt --steps 600 --block-size 40 --save twin.json
  python3 microgpt_lab.py --load twin.json --chat

元コード: https://github.com/karpathy (microgpt)
"""

import argparse
import json
import math
import random
import time


# ---- Autograd(microgpt.py と同じ) ----
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad


def main():
    ap = argparse.ArgumentParser(description="microgpt experiment lab")
    ap.add_argument("--input", default="input.txt", help="学習データ(1行=1ドキュメント)")
    ap.add_argument("--steps", type=int, default=1000, help="訓練ステップ数(既定: 1000)")
    ap.add_argument("--block-size", type=int, default=16, help="文脈長。行の最初の何文字まで学べるか(既定: 16)")
    ap.add_argument("--n-layer", type=int, default=1, help="層の数(既定: 1)")
    ap.add_argument("--n-embd", type=int, default=16, help="埋め込みの幅。4の倍数(既定: 16)")
    ap.add_argument("--temperatures", type=float, nargs="+", default=[0.5], help="サンプル生成の温度(複数可)")
    ap.add_argument("--num-samples", type=int, default=10, help="温度ごとのサンプル数(既定: 10)")
    ap.add_argument("--seed", type=int, default=42, help="乱数シード(既定: 42)")
    ap.add_argument("--chat", action="store_true", help="訓練後に対話モードに入る。文の出だしを入力すると双子が続きを書く")
    ap.add_argument("--save", default=None, help="訓練後にモデルを保存するJSONファイル(例: twin.json)")
    ap.add_argument("--load", default=None, help="保存済みモデルを読み込み、訓練せずに生成・チャットする(例: twin.json)")
    args = ap.parse_args()

    random.seed(args.seed)
    n_head = 4

    if args.load:
        # 保存済みモデルを読み込む。データ読み込みと訓練はスキップして、生成・チャットへ。
        with open(args.load, encoding="utf-8") as f:
            ckpt = json.load(f)
        n_layer, n_embd, block_size = ckpt["n_layer"], ckpt["n_embd"], ckpt["block_size"]
        uchars = ckpt["uchars"]
        state_dict = {k: [[Value(x) for x in row] for row in mat] for k, mat in ckpt["state_dict"].items()}
        params, docs = [], []
        args.steps = 0  # 訓練ループは回さない
        print(f"loaded model: {args.load} | vocab: {len(uchars) + 1} | block_size: {block_size}")
    else:
        n_layer, n_embd, block_size = args.n_layer, args.n_embd, args.block_size
        # データセット: 1行 = 1ドキュメント
        docs = [line.strip() for line in open(args.input, encoding="utf-8") if line.strip()]
        random.shuffle(docs)
        uchars = sorted(set("".join(docs)))
        print(f"docs: {len(docs)} | vocab: {len(uchars) + 1} | block_size: {block_size}")
        if len(uchars) + 1 > 200:
            print("warning: 語彙(ユニーク文字数)が多いので訓練がかなり遅くなります。"
                  "データをひらがな化・ASCII化すると速くなります")
        # パラメータ初期化(microgpt.py と同じ構成)
        matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
        state_dict = {'wte': matrix(len(uchars) + 1, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(len(uchars) + 1, n_embd)}
        for i in range(n_layer):
            state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
            state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
            state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
            state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
            state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
            state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
        params = [p for mat in state_dict.values() for row in mat for p in row]
        print(f"params: {len(params)}")

    assert n_embd % n_head == 0, "--n-embd は 4 の倍数にしてください"
    head_dim = n_embd // n_head
    stoi = {ch: i for i, ch in enumerate(uchars)}
    BOS = len(uchars)
    vocab_size = len(uchars) + 1

    def linear(x, w):
        return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

    def softmax(logits):
        max_val = max(val.data for val in logits)
        exps = [(val - max_val).exp() for val in logits]
        total = sum(exps)
        return [e / total for e in exps]

    def rmsnorm(x):
        ms = sum(xi * xi for xi in x) / len(x)
        scale = (ms + 1e-5) ** -0.5
        return [xi * scale for xi in x]

    def gpt(token_id, pos_id, keys, values):
        tok_emb = state_dict['wte'][token_id]
        pos_emb = state_dict['wpe'][pos_id]
        x = [t + p for t, p in zip(tok_emb, pos_emb)]
        x = rmsnorm(x)
        for li in range(n_layer):
            x_residual = x
            x = rmsnorm(x)
            q = linear(x, state_dict[f'layer{li}.attn_wq'])
            k = linear(x, state_dict[f'layer{li}.attn_wk'])
            v = linear(x, state_dict[f'layer{li}.attn_wv'])
            keys[li].append(k)
            values[li].append(v)
            x_attn = []
            for h in range(n_head):
                hs = h * head_dim
                q_h = q[hs:hs+head_dim]
                k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
                v_h = [vi[hs:hs+head_dim] for vi in values[li]]
                attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
                attn_weights = softmax(attn_logits)
                head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
                x_attn.extend(head_out)
            x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
            x = [a + b for a, b in zip(x, x_residual)]
            x_residual = x
            x = rmsnorm(x)
            x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
            x = [xi.relu() for xi in x]
            x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
            x = [a + b for a, b in zip(x, x_residual)]
        logits = linear(x, state_dict['lm_head'])
        return logits

    # Adam オプティマイザ
    learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
    m = [0.0] * len(params)
    v = [0.0] * len(params)

    # 訓練ループ
    num_steps = args.steps
    t0 = time.time()
    report_every = max(1, num_steps // 10)
    for step in range(num_steps):
        doc = docs[step % len(docs)]
        tokens = [BOS] + [stoi[ch] for ch in doc] + [BOS]
        n = min(block_size, len(tokens) - 1)

        keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
        losses = []
        for pos_id in range(n):
            token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
            logits = gpt(token_id, pos_id, keys, values)
            probs = softmax(logits)
            loss_t = -probs[target_id].log()
            losses.append(loss_t)
        loss = (1 / n) * sum(losses)
        loss.backward()

        lr_t = learning_rate * (1 - step / num_steps)
        for i, p in enumerate(params):
            m[i] = beta1 * m[i] + (1 - beta1) * p.grad
            v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
            m_hat = m[i] / (1 - beta1 ** (step + 1))
            v_hat = v[i] / (1 - beta2 ** (step + 1))
            p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
            p.grad = 0

        if (step + 1) % report_every == 0 or step == 0:
            elapsed = time.time() - t0
            print(f"step {step+1:5d} / {num_steps} | loss {loss.data:.4f} | {elapsed:6.1f}s")

    # 保存: 重みと語彙をJSONに書き出す(次回 --load で訓練なしに使える)
    if args.save:
        ckpt = {
            "n_layer": n_layer, "n_embd": n_embd, "block_size": block_size,
            "uchars": uchars,
            "state_dict": {k: [[p.data for p in row] for row in mat] for k, mat in state_dict.items()},
        }
        with open(args.save, "w", encoding="utf-8") as f:
            json.dump(ckpt, f)
        print(f"saved model: {args.save}")

    # 生成: 温度ごとにサンプルを出力
    for temperature in args.temperatures:
        print(f"\n--- temperature {temperature} ---")
        for sample_idx in range(args.num_samples):
            keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
            token_id = BOS
            sample = []
            for pos_id in range(block_size):
                logits = gpt(token_id, pos_id, keys, values)
                probs = softmax([l / temperature for l in logits])
                token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
                if token_id == BOS:
                    break
                sample.append(uchars[token_id])
            print(f"sample {sample_idx+1:2d}: {''.join(sample)}")

    # 対話: 学習した「双子」に文の出だしを与え、続きを書かせる
    # (本物の対話モデルではない。文字レベルの「次の1文字予測」で続きを作るだけ)
    if args.chat:
        temperature = args.temperatures[0]
        print(f"\n--- あなたの双子と話す / chat with your twin (temperature {temperature}) ---")
        print("文の出だしを入力すると、双子が「君の口調」で続きを書きます。")
        print("Type the start of a line; your twin finishes it in your voice.")
        print("空行か Ctrl-C で終了 / empty line or Ctrl-C to quit.\n")
        while True:
            try:
                seed = input("you  > ")
            except (EOFError, KeyboardInterrupt):
                print()
                break
            if seed.strip() == "":
                break
            keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
            seed_ids = [stoi[ch] for ch in seed if ch in stoi]
            logits, pos_id = None, 0
            for tok in [BOS] + seed_ids:   # BOS と入力文字を順に食わせて文脈を作る
                if pos_id >= block_size:
                    break
                logits = gpt(tok, pos_id, keys, values)
                pos_id += 1
            twin = []
            while logits is not None and pos_id < block_size:
                probs = softmax([l / temperature for l in logits])
                token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
                if token_id == BOS:
                    break
                twin.append(uchars[token_id])
                logits = gpt(token_id, pos_id, keys, values)
                pos_id += 1
            print(f"twin > {seed}{''.join(twin)}\n")


if __name__ == "__main__":
    main()


### 実験A — 温度(temperature)で遊ぶ / Play with temperature

生成のときの「冒険度」のつまみです。
- **低い(0.1)** = 安全運転。いちばん確率の高い文字ばかり選ぶ
- **高い(2.0)** = 大冒険。確率の低い文字もどんどん選ぶ

ChatGPT にも同じつまみがあります。1回の訓練で、4つの温度のサンプルをまとめて出します。

In [ ]:
!python3 microgpt_lab.py --input input.txt --steps 600 --temperatures 0.1 0.5 1.0 2.0 --num-samples 5

**問い / Questions**
- 温度 0.1 の名前、おたがい似すぎていない?
- 温度 2.0 では何が起きている?
- 「ちょうどいい創造性」は何度くらい? それは作るものによって変わる?

## Step 4 — データセット・メニュー / Dataset menu

同じ 200 行のコードが、**データを変えるだけで別の専門家**になります。3つのデータから1つ選んでください(全部やってもOK)。

| メニュー | データ | 見どころ |
|---|---|---|
| A | フォーク民謡(ABC記譜法) | AI が作曲する。楽譜は音にして聞ける |
| B | チェスの棋譜 | ルールを教えていないのに「合法っぽい手」を指す |
| C | 俳句(ひらがな) | 5-7-5 を守れるか、指を折って数えよう |

**データの入れ方**: 下のセルを実行すると、3つのデータセットが自動でダウンロードされます(アップロード不要)。

**自分で操る**: メニューを試したあと、下の「**自分で操る — 対話する**」セルでモデルと対話できます(コマンド末尾の `--chat`)。ただしモデルは意味を理解せず、打った**出だしの続き**を書くだけです。

In [ ]:
# データセットを GitHub から自動ダウンロード / Auto-download the datasets
import urllib.request
BASE = "https://raw.githubusercontent.com/itoksk/summer-ai-materials/main/materials/data/"
for name in ["input_abc.txt", "input_chess.txt", "input_haiku.txt"]:
    urllib.request.urlretrieve(BASE + name, name)
    n = sum(1 for _ in open(name, encoding="utf-8"))
    print(f"{name}: {n} 行 / lines  ダウンロード完了 / done")

### メニューA — AI が作曲する / ABC folk tunes

データはフォーク民謡(Nottingham Music Database)の楽譜を ABC 記譜法という文字の形式にしたものです。下のセルを実行すると、AI がメロディーを生成します。

In [ ]:
!python3 microgpt_lab.py --input input_abc.txt --steps 400 --block-size 48 --temperatures 0.7 --num-samples 5

**音にして聞く / Hear it**
1. 気に入ったサンプル(`K:` や `|` が入った行)をコピー
2. [editor.drawthedots.com](https://editor.drawthedots.com) を開く
3. 次のテンプレートの最後の行に貼り付けると、再生ボタンで聞けます:

```
X:1
T:AI Tune
M:6/8
L:1/8
K:Gmaj
(ここに生成されたメロディーを貼る)
```

変な音になったら、それも「発見」。別のサンプルでもう一度。

### メニューB — チェスを「見て」覚える / Chess openings

データは「1. e4 c5 2. Nf3 ...」のような対局の出だしです。AI にチェスのルールは**一切教えていません**。

In [ ]:
!python3 microgpt_lab.py --input input_chess.txt --steps 500 --block-size 40 --temperatures 0.5 --num-samples 5

**検証 / Verify**: 生成された棋譜を [lichess.org/analysis](https://lichess.org/analysis) に貼って、**何手目まで合法か**数えてみよう。

ルールを教えていないのに、なぜ合法手っぽくなる? — 「次の文字を当てる」だけで、世界の決まりごとがしみ込んでいく。これが大きな LLM で起きていることの縮図です。

### メニューC — 俳句 5-7-5 / Haiku

データはひらがなの俳句です(漢字だと文字の種類が多すぎて、この小さな AI には大変なので)。

In [ ]:
!python3 microgpt_lab.py --input input_haiku.txt --steps 500 --block-size 24 --temperatures 0.8 --num-samples 8

**数えてみよう / Count**: 生成された句を、指を折って 5-7-5 で数えてみよう。
- 何句が 5-7-5 になった?
- ChatGPT も実は「数える」のが苦手です(strawberry に r は何個? 問題)。理由はトークンの仕組みにあります — コースサイトのトークナイザーのページとつながる話

### 自分で操る — 対話する / Drive it yourself (chat)

サンプルを眺めるだけでなく、自分で**ハンドルを握れます**。下のセルを実行すると、訓練のあと「`you >`」の入力欄が出ます。**出だし**を打つと、モデルがその続きを書きます。

たいせつ: モデルは**意味を理解していません**。俳句モデルに「桜」と打っても、桜を**お題にした句を作る**のではなく、桜の後に続きそうな文字を並べるだけです。チェスは序盤の数手、俳句は上の句、音楽は数音を打ってみましょう。**空行**で終了。

(お題で本当に一句詠ませたいときは、指示を理解する大きいモデル=Gem の出番です — これが「スケール」の差。)

In [ ]:
# 音楽と話す / chat with the music model
!python3 microgpt_lab.py --input input_abc.txt --steps 400 --block-size 48 --chat

In [ ]:
# チェスと話す / chat with the chess model
!python3 microgpt_lab.py --input input_chess.txt --steps 500 --block-size 40 --chat

In [ ]:
# 俳句と話す / chat with the haiku model
!python3 microgpt_lab.py --input input_haiku.txt --steps 500 --block-size 24 --chat

### Show & Tell

チームで共有しよう:
- **ベストの1つ**(いちばん「らしい」生成)
- **最大の失敗作**(失敗はネタとして最高。なぜ失敗したか考えるのが本番)

## Step 5 — 自分データGPT / My-Data GPT

最後は **自分のことば** で訓練します。

1. 先生から共有された **ヒアリングGem** を開いてチャットする(インタビューに答える)
2. 最低 5 問は答えてから「コーパスを作って / make my corpus」と頼む
3. 出てきたコードブロックの中身を `my_corpus.txt` という名前で保存
4. このColabにアップロード(左のフォルダにドラッグ)

**たいせつ**: 本名・住所・学校名・連絡先はコーパスに入れない。保存する前に自分で読み直そう。これが「学習データ」になります。

In [ ]:
%%writefile gem_to_corpus.py
#!/usr/bin/env python3
"""ヒアリングGemの出力を microgpt 用の input.txt に変換するスクリプト。

Gem(materials/gems/hearing-gem.md)が出力したコーパス(1行=1文)を受け取り、
microgpt.py がそのまま学習できる形に整える:

  - コードブロック記号・番号・引用符などの飾りを除去
  - Unicode 正規化(NFC)と空白の整理
  - 個人情報らしき行(URL / メール / 電話番号)を除外して警告
  - 長すぎる行を切り詰め(文字レベルGPTは短い行ほど速く学べる)
  - 重複行を除去
  - 行数が少なすぎる場合は --augment でシャッフル反復して水増し
    (microgpt は学習中にデータを自動で周回するので必須ではない)

使い方:
  python3 gem_to_corpus.py my_corpus.txt
  python3 gem_to_corpus.py my_corpus.txt -o input.txt --max-chars 40 --augment

依存ライブラリなし(Python 3.8+ の標準ライブラリのみ)。
"""

import argparse
import random
import re
import statistics
import sys
import unicodedata

# 個人情報・ノイズの検出パターン(該当行はコーパスから外して警告する)
RE_URL = re.compile(r"https?://|www\.")
RE_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
RE_PHONE = re.compile(r"\d{2,4}[-\s]?\d{2,4}[-\s]?\d{3,4}")
# 行頭の飾り: 番号("1. " "12) ")、箇条書き記号、引用記号
RE_DECOR = re.compile(r"^\s*(?:\d{1,3}[.)]\s*|[-*>・]\s*)")


def clean_line(line: str) -> str:
    line = unicodedata.normalize("NFC", line)
    line = RE_DECOR.sub("", line)
    line = line.strip().strip('"“”「」').strip()
    line = re.sub(r"\s+", " ", line)
    return line


def truncate(line: str, max_chars: int) -> str:
    if len(line) <= max_chars:
        return line
    cut = line[:max_chars]
    # なるべく単語・文節の切れ目(空白か句読点)で切る
    for i in range(len(cut) - 1, max(0, max_chars - 12), -1):
        if cut[i] in " 、。!?！？,.":
            return cut[: i + 1].rstrip()
    return cut


def extract_lines(text: str) -> list[str]:
    """フェンス付きコードブロックがあればその中身を、なければ全文を行リストにする。"""
    blocks = re.findall(r"```[\w-]*\n(.*?)```", text, flags=re.DOTALL)
    body = "\n".join(blocks) if blocks else text
    return body.splitlines()


def main() -> int:
    ap = argparse.ArgumentParser(description="Gem corpus -> microgpt input.txt")
    ap.add_argument("source", help="Gemの出力を保存したテキストファイル(例: my_corpus.txt)")
    ap.add_argument("-o", "--output", default="input.txt", help="出力先(既定: input.txt)")
    ap.add_argument("--max-chars", type=int, default=40, help="1行の最大文字数(既定: 40)")
    ap.add_argument("--min-chars", type=int, default=4, help="これより短い行は捨てる(既定: 4)")
    ap.add_argument("--min-lines", type=int, default=150, help="--augment 時の目標行数(既定: 150)")
    ap.add_argument("--augment", action="store_true", help="行数が足りない場合シャッフル反復で水増しする")
    ap.add_argument("--seed", type=int, default=42, help="--augment 用の乱数シード")
    args = ap.parse_args()

    try:
        text = open(args.source, encoding="utf-8").read()
    except OSError as e:
        print(f"error: {args.source} を読めません: {e}", file=sys.stderr)
        return 1

    kept: list[str] = []
    dropped_pii = 0
    seen = set()
    for raw in extract_lines(text):
        line = clean_line(raw)
        if len(line) < args.min_chars:
            continue
        if RE_URL.search(line) or RE_EMAIL.search(line) or RE_PHONE.search(line):
            dropped_pii += 1
            continue
        line = truncate(line, args.max_chars)
        if line in seen:
            continue
        seen.add(line)
        kept.append(line)

    if not kept:
        print("error: 使える行が1行もありません。Gemの出力を確認してください。", file=sys.stderr)
        return 1

    corpus = list(kept)
    if args.augment and len(corpus) < args.min_lines:
        rng = random.Random(args.seed)
        while len(corpus) < args.min_lines:
            batch = list(kept)
            rng.shuffle(batch)
            corpus.extend(batch)
        corpus = corpus[: max(args.min_lines, len(kept))]

    with open(args.output, "w", encoding="utf-8") as f:
        f.write("\n".join(corpus) + "\n")

    lengths = [len(l) for l in kept]
    vocab = sorted(set("".join(kept)))
    print(f"wrote {args.output}: {len(corpus)} lines ({len(kept)} unique)")
    print(f"  line length  min {min(lengths)} / median {int(statistics.median(lengths))} / max {max(lengths)}")
    print(f"  vocab size   {len(vocab) + 1} (unique chars + BOS)")
    if dropped_pii:
        print(f"  note: URL・メール・電話番号らしき {dropped_pii} 行を除外しました")
    if len(kept) < 80:
        print("  warning: 行数が少なめです。Gemにもう少しインタビューに答えてから再出力させると良くなります")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# Gemの出力 my_corpus.txt を、訓練用の input_me.txt に変換する
!python3 gem_to_corpus.py my_corpus.txt -o input_me.txt --augment

In [ ]:
!python3 microgpt_lab.py --input input_me.txt --steps 600 --block-size 40 --temperatures 0.8 --num-samples 10

### 双子と話す / Chat with your twin

下のセルを実行すると、訓練のあと **入力欄** が出ます。**文の出だし**(例:「わたしは」「いつも」)を打つと、双子が君の口調で **続き** を書きます。

たいせつ: これは本物のチャットボットではありません。**次の1文字を予測しているだけ**なので、返ってくるのは質問への「答え」ではなく、君が打った文の『続き』です。これが今日の学びそのもの — 中身は理解していないのに、口調だけは君っぽい。**空行**を入れると終了します。

In [ ]:
!python3 microgpt_lab.py --input input_me.txt --steps 600 --block-size 40 --temperatures 0.8 --num-samples 3 --chat

### 品評会 / Show & Tell

- 「**一番自分っぽい一言**」と「**一番ハズした一言**」を1つずつ発表
- 問い: なぜ 150 行のデータでは「自分」を完全に学べないのだろう? ChatGPT は何兆文字で訓練されている? — **データの量と質がAIのすべてを決める**

## ふりかえり / Reflection

- 「次の1文字を当てる」だけのゲームから、何が生まれた?
- データを変えると AI はどう変わった? 温度を変えると?
- 自分GPTは「自分」だった? どこが違った?

**もっとやりたい人へ / Challenges**
- `--steps 2000` にすると? (時間とひきかえに賢くなる?)
- `--n-embd 32` で脳を大きくすると?
- 2つのデータファイルを混ぜて訓練したら?
- 自分のパソコンで本物のローカルLLM(Ollama)を動かす — コースサイトの Day 2 ページへ